# NB4: Pydantic Type-Safe Delegation

**2-minute intro script:** Manager agents often pass raw text to specialists. That is fine for a demo, but fragile in production. If a Data Analyst hallucinates a field or a Coder returns an unsafe file path, downstream agents may crash or do the wrong thing. Pydantic turns each handoff into an enforceable contract. This notebook shows valid handoffs, invalid extra fields, unsafe patch paths, and a full PM -> Tech Lead -> Coder -> QA -> Reviewer chain.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator
from typing import List, Literal

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class TaskSpec(StrictModel):
    goal: str
    acceptance_criteria: List[str] = Field(min_length=1)
    risk: Literal["low", "medium", "high"]
    priority: int = Field(ge=1, le=5)

class CodePatch(StrictModel):
    files: dict[str, str]
    rationale: str
    tests_to_run: List[str] = Field(min_length=1)

    @field_validator("files")
    @classmethod
    def validate_paths(cls, files):
        unsafe_paths = ["/etc/", "../", "~/"]
        for path in files.keys():
            if any(unsafe in path for unsafe in unsafe_paths):
                raise ValueError(f"Unsafe path: {path}")
        return files

class TestResult(StrictModel):
    passed: bool
    log: str
    failing_tests: List[str] = Field(default_factory=list)
    coverage_percent: float = Field(ge=0.0, le=100.0)

class ReviewDecision(StrictModel):
    approved: bool
    requires_changes: List[str] = Field(default_factory=list)
    security_notes: str | None = None

In [ ]:
def demo_schema_enforcement():
    print("=== Example 1: Valid TaskSpec ===")
    task = TaskSpec(
        goal="Create a slugify function",
        acceptance_criteria=["Handles spaces", "Handles special chars", "Has tests"],
        risk="low",
        priority=3,
    )
    print(task.model_dump_json(indent=2))

    print("\n=== Example 2: Invalid TaskSpec (extra field) ===")
    try:
        TaskSpec(
            goal="Create a slugify function",
            acceptance_criteria=["Has tests"],
            risk="low",
            priority=3,
            secret_field="should_fail",
        )
    except ValidationError as exc:
        print("Caught:", exc.errors()[0]["msg"])

    print("\n=== Example 3: Invalid CodePatch (unsafe path) ===")
    try:
        CodePatch(
            files={"../../../etc/passwd": "malicious content"},
            rationale="Adding config",
            tests_to_run=["test_config"],
        )
    except ValidationError as exc:
        print("Caught:", exc.errors()[0]["msg"])

demo_schema_enforcement()

In [ ]:
def demo_valid_handoff_chain():
    task = TaskSpec(
        goal="Add user authentication",
        acceptance_criteria=["JWT tokens", "Password hashing", "Rate limiting"],
        risk="high",
        priority=1,
    )
    patch = CodePatch(
        files={"auth.py": "def login(): ...", "tests_auth.py": "def test_login(): ..."},
        rationale="Implemented JWT-based auth",
        tests_to_run=["test_login", "test_logout", "test_token_refresh"],
    )
    result = TestResult(
        passed=True,
        log="All tests passed",
        failing_tests=[],
        coverage_percent=87.5,
    )
    decision = ReviewDecision(
        approved=True,
        requires_changes=[],
        security_notes="JWT secret must be rotated monthly",
    )

    print("Full chain: PM -> Tech Lead -> Coder -> QA -> Reviewer -> Release")
    print(f"Task: {task.goal}")
    print(f"Files: {list(patch.files.keys())}")
    print(f"Tests: {result.passed} ({result.coverage_percent}% coverage)")
    print(f"Decision: {'APPROVED' if decision.approved else 'REJECTED'}")

demo_valid_handoff_chain()

## The Missing Brain: Mock LLM Adapter

The previous cells proved the deterministic governance layer. Now we simulate the messy part: an LLM that returns malformed JSON. The model is not trusted. The schema is trusted. This is the bridge from a deterministic state machine to a managed agentic AI system.

In [ ]:
def mock_llm_agent(task: str, attempt: int = 0) -> dict:
    """Simulate an LLM generating a CodePatch.

    Attempt 0 hallucinates an extra field and unsafe path.
    Attempt 1 repairs the JSON after receiving validation feedback.
    """
    if attempt == 0:
        return {
            "files": {"../escape.py": "print('hack')"},
            "rationale": f"I am adding config for: {task}",
            "tests_to_run": ["test_config"],
            "hallucinated_field": "I am an LLM and I made this up.",
        }
    return {
        "files": {"math.py": "def add(a, b): return a + b"},
        "rationale": "Fixed the path and removed extra fields based on validation error.",
        "tests_to_run": ["test_add"],
    }

print("--- Attempt 1: LLM hallucinates ---")
validation_feedback = ""
try:
    CodePatch.model_validate(mock_llm_agent("implement add(a, b)", attempt=0))
except ValidationError as exc:
    validation_feedback = str(exc.errors())
    print("Pydantic caught the LLM hallucination:")
    for error in exc.errors():
        print(f"- {error['loc']}: {error['msg']}")
    print("Triggering repair loop...\n")

print("--- Attempt 2: LLM repairs ---")
valid_patch = CodePatch.model_validate(mock_llm_agent(validation_feedback, attempt=1))
print("LLM output validated successfully:")
print(valid_patch.model_dump_json(indent=2))

## Framework Mapping

In production frameworks, you should keep the same mental model: the LLM produces candidate output, and the framework validates it against a schema before another agent or tool consumes it.

**CrewAI mapping:**

```python
from crewai import Agent, Task, Crew

coder = Agent(role="Coder", goal="Write code", backstory="...")

coding_task = Task(
    description="Write the code",
    expected_output="A valid CodePatch",
    agent=coder,
    output_pydantic=CodePatch,  # schema enforcement
)
```

**LangGraph mapping:**

```python
from typing_extensions import TypedDict

class AgentState(TypedDict):
    task: TaskSpec
    patch: CodePatch | None
    test_result: TestResult | None
    repair_attempts: int
```

The custom Python version teaches what the framework is doing for you: validate, reject, repair, and only then route forward.

## 🧪 Exercises: The Hallucination Firewall

**The Story:** Your Requirements Analyst agent is useful, but it hallucinates. Today it added a `"priority_vibes": "urgent-ish"` field to the handoff. Your Implementation Planner crashes because it expects a typed priority and a bounded risk value. The whole pipeline dies. How do we catch the hallucination before it breaks the system?

**Your Mission:**
1. **The Release Truth:** Add a `ReleaseRequest` schema for the software-delivery baseline. It must include `service_name`, `requested_change`, and `priority`.
2. **The Risk Guardrail:** Add a `ReleasePlan` schema. It must include `deployment_window`, `rollback_plan`, and `risk_level`. Use Pydantic validators to reject high-risk releases with an empty rollback plan.
3. **The Security Gatekeeper:** Make `ReviewDecision` reject high-risk tasks unless `security_notes` is present and not empty.
4. **The Gauntlet:** Write one valid and one invalid JSON payload for each schema. Prove that the invalid payloads are caught by `extra="forbid"` or field validators.
5. **The Feedback Loop:** Convert a `ValidationError` into a structured feedback string that can be fed back into the LLM's prompt for a repair attempt.

**The Takeaway:** We do not trust the LLM to be perfect. We trust the schema to catch the imperfection. Pydantic is the bouncer at the door of your AI workforce.

In [ ]:
# ==========================================
# LIVE LLM EXECUTION (Optional)
# ==========================================
# The cells above run offline using deterministic mocks.
# To see a real LLM generate output constrained by this schema:
#
#   pip install openai instructor
#   export OPENAI_API_KEY="..."
#
# Keep this False for workshops unless learners have API keys.
USE_LIVE_LLM = False

if USE_LIVE_LLM:
    import os
    import instructor
    from openai import OpenAI

    client = instructor.from_openai(OpenAI(api_key=os.environ["OPENAI_API_KEY"]))
    model_name = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

    live_result = client.chat.completions.create(
        model=model_name,
        response_model=CodePatch,
        messages=[
            {
                "role": "user",
                "content": 'Return a safe CodePatch that implements add(a, b) in math.py with one test named test_add.',
            }
        ],
    )
    print(live_result.model_dump_json(indent=2))